# PTB-XL pretraining, then MIT-BIH fine-tuning

One run does all of it for each chosen input variant:

1. preprocess PTB-XL (once per session),
2. pretrain the model on PTB-XL beats (3 seeds),
3. fine-tune on MIT-BIH DS1 from the pretrained weights (6 runs),
4. train the same model from scratch for comparison (6 runs).

The letters are the same four input variants as in Module 1 (A, B, C, D).

Before running (details in `kaggle/README.md`):

1. Settings: **Internet on**. Accelerator: **GPU**.
2. Add-ons > Secrets: `GITHUB_TOKEN` (read access to the repo), ticked for this notebook.
3. Add Input: the team's `mitdb-raw` dataset.
   PTB-XL comes from PhysioNet by default (1.8 GB download). To use an attached Kaggle copy instead,
   add it as an input and set `PTBXL_SOURCE = "/kaggle/input"`.
4. Set the parameters in the next cell, then Run All.
5. Download `ecgaug_<sha>_<time>.zip` from Output and run `python kaggle/ingest_runs.py <zip>` in your clone.

In [ ]:
# --- Parameters: edit only this cell ---------------------------------------------------------
VARIANTS = ["A"]                      # input variants to run: any of "A", "B", "C", "D"
PTBXL_SOURCE = "download"             # "download" = fetch PTB-XL 1.0.3 from PhysioNet; or a folder such as "/kaggle/input"
SMOKE_TEST = False                    # True = quick check on fake data (about a minute, no results)
GIT_REF = "main"                      # branch name or full commit SHA to run
DEVICE = "auto"                       # "auto" uses the GPU when the session has one
ENV_MODE = "locked"                   # "locked" = exact uv.lock versions; "kaggle" = preinstalled packages (faster, debugging only)
RAW_DATASET_DIR = "/kaggle/input"     # the attached mitdb-raw dataset; "" downloads MIT-BIH from PhysioNet
REPO_URL = "https://github.com/akashsaha477/Gen_Ai_Project.git"

In [ ]:
import os, subprocess, sys
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "repo"
env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = None                      # public repository, or the secret is not attached
if token:
    askpass = WORK / ".git-askpass.sh"
    askpass.write_text('#!/bin/sh\necho "$GITHUB_TOKEN"\n')
    askpass.chmod(0o700)
    env.update(GIT_ASKPASS=str(askpass), GITHUB_TOKEN=token)   # the token is never printed or put in a URL
    url = REPO_URL.replace("https://", "https://x-access-token@")

if not REPO.exists():
    subprocess.run(["git", "clone", "--quiet", url, str(REPO)], check=True, env=env)
subprocess.run(["git", "fetch", "--quiet", "origin"], cwd=REPO, check=True, env=env)
ref = GIT_REF if len(GIT_REF) == 40 else f"origin/{GIT_REF}"
subprocess.run(["git", "checkout", "--quiet", "--detach", ref], cwd=REPO, check=True)
sha = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True).stdout.strip()
print("running commit", sha)

In [ ]:
# Turn the variant letters into config files, e.g. "A" -> configs/ptbxl_pretraining/A_short_window.yaml
configs = []
for letter in VARIANTS:
    (path,) = sorted((REPO / "configs/ptbxl_pretraining").glob(f"{letter}_*.yaml"))
    configs.append(str(path.relative_to(REPO)))
print("configs:", configs)

extra = ["--device", DEVICE] + (["--ptbxl-download"] if PTBXL_SOURCE == "download" else ["--ptbxl-root", PTBXL_SOURCE])
if SMOKE_TEST:
    extra.append("--smoke")
has_raw = RAW_DATASET_DIR and any(Path(RAW_DATASET_DIR).rglob("100.dat"))
cmd = [sys.executable, "kaggle/kaggle_run.py", "--cli", "ecgaug.experiments.ptbxl_pretraining", "--env", ENV_MODE, "--out", str(WORK),
       "--config", *configs]
if has_raw:
    cmd += ["--raw-dir", RAW_DATASET_DIR]
cmd += ["--", *extra]
code = subprocess.run(cmd, cwd=REPO).returncode
print("finished with exit code", code, "(2 = the run was refused, read the message above)")